# Objectifs

Le CSV ne possède pas d'en-têtes et certaines lignes peuvent mal se comporter au chargement. Je dois donc : 
1. compter toutes les lignes physiques du fichier;
2. charger les lignes correctements structurées;
3. isoler explicitement les lignes problématiques;
4. expliquer le problème; sans effacer quoique ce soit silencieusement. 

Le fichier est censé avoir 11 colonnes, dans un ordre donné dans l'énoncé.

## I. Imports et constantes

In [11]:
from pathlib import Path
import csv
import urllib.request
import pandas as pd

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

url = (
    "https://raw.githubusercontent.com/planetsig/ufo-reports/master/csv-data/ufo-complete-geocoded-time-standardized.csv"
)

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

## II. Téléchargement du fichier

In [12]:
if not DATA_PATH.exists():
    print(f"Downloading data from {url}...")
    urllib.request.urlretrieve(url, DATA_PATH)
    print(f"Data downloaded to {DATA_PATH}")

"urllib.request" permet d'éviter l'ajout d'une dépendance inutile contrairement à "request"

## III. Comptage des lignes physiques

In [13]:
with open(DATA_PATH, "r", encoding="utf-8", errors="replace") as f:
    nb_lignes_fichier = sum(1 for _ in f)

print(f"Nombre de lignes dans le fichier: {nb_lignes_fichier}")

Nombre de lignes dans le fichier: 88875


## IV. Lecture contrôlée avec csv.reader

In [14]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)

    for numero_ligne, ligne in enumerate(reader, start=1):
        if len(ligne) == len(COLUMNS):
            lignes_valides.append(ligne)
        else:
            lignes_problemes.append({
                "numero_ligne" : numero_ligne,
                "nb_champs" : len(ligne),
                "contenu" : ligne
            })

df_raw = pd.DataFrame(lignes_valides, columns=COLUMNS)
df_problemes_chargement = pd.DataFrame(lignes_problemes)

print("Lignes physiques du fichier:", nb_lignes_fichier)
print("Lignes chargées :", len(df_raw))
print("Lignes traitées à part :", len(df_problemes_chargement))
print("Vérification :", len(df_raw) + len(df_problemes_chargement))

Lignes physiques du fichier: 88875
Lignes chargées : 88679
Lignes traitées à part : 196
Vérification : 88875


## V. Afficher une anomalie de structure

In [15]:
df_problemes_chargement.head()

,numero_ligne,nb_champs,contenu
0,877,12,"[10/1/2006 12:00, , , , , 0, , , ((EDITORIAL C..."
1,1712,12,"[10/14/2004 13:00, , , , , 0, , , With all the..."
2,1814,12,"[10/14/2011 22:30, , nv, , , 0, light, 22, 3 G..."
3,2857,12,"[10/17/2008 20:30, , tx, , , 0, oval, 5 minute..."
4,3733,12,"[10/20/2013 18:30, , ct, , , 0, egg, 2 hours, ..."


In [16]:
if not df_problemes_chargement.empty:
    probleme = df_problemes_chargement.iloc[0]
    print("Numero de ligne :", probleme["numero_ligne"])
    print("Nombre de champs trouvés :", probleme["nb_champs"])
    print("Contenu :", probleme["contenu"])

Numero de ligne : 877
Nombre de champs trouvés : 12
Contenu : ['10/1/2006 12:00', '', '', '', '', '0', '', '', '((EDITORIAL COMMENT ABOUT THE UFO PHENOMEN))  ufo+alien+reptiles', '10/30/2006', '0', '0']


In [17]:
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df_problemes_chargement.to_csv(
    OUTPUT_DIR / "lignes_problemes_chargement.csv",
    index=False
)